In [2]:
# ============================================================
# SINGLE CELL: Robustly find saved Kaggle notebook artifacts
# and benchmark GPU latency
# ============================================================

import os
import time
from pathlib import Path

import torch
import torch.nn as nn
from torchvision import models
import pandas as pd

# -----------------------------
# SETTINGS
# -----------------------------
NUM_CLASSES = 14
IMAGE_SIZE = 224
BATCH_SIZE = 1
WARMUP = 30
RUNS = 200

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -----------------------------
# DEBUG: SHOW WHAT IS MOUNTED
# -----------------------------
input_root = Path("/kaggle/input")
working_root = Path("/kaggle/working")

print("\n========== DIRECTORY DEBUG ==========")
print("Does /kaggle/input exist?   ", input_root.exists())
print("Does /kaggle/working exist? ", working_root.exists())

if input_root.exists():
    print("\nTop-level folders under /kaggle/input:")
    top_dirs = sorted(list(input_root.iterdir()))
    if not top_dirs:
        print("  [EMPTY]")
    else:
        for p in top_dirs:
            print(" ", p)
            try:
                children = sorted(list(p.iterdir()))[:10]
                for c in children:
                    print("     -", c.name)
            except Exception as e:
                print("     [Could not list contents]", e)

print("\nTop-level folders under /kaggle/working:")
if working_root.exists():
    try:
        for p in sorted(list(working_root.iterdir()))[:20]:
            print(" ", p)
    except Exception as e:
        print("  [Could not list /kaggle/working]", e)

# -----------------------------
# ROBUST RECURSIVE SEARCH
# -----------------------------
def recursive_find_checkpoints():
    """
    Search recursively for the expected checkpoint files under
    /kaggle/input and /kaggle/working.
    """
    search_roots = []
    if input_root.exists():
        search_roots.append(input_root)
    if working_root.exists():
        search_roots.append(working_root)

    all_pth = []
    all_npy = []
    candidate_ckpt_dirs = set()

    for root in search_roots:
        try:
            all_pth.extend(root.rglob("*.pth"))
            all_npy.extend(root.rglob("*.npy"))
        except Exception as e:
            print(f"Could not scan {root}: {e}")

    for p in all_pth:
        candidate_ckpt_dirs.add(p.parent)

    return sorted(all_pth), sorted(all_npy), sorted(candidate_ckpt_dirs)

all_pth, all_npy, candidate_ckpt_dirs = recursive_find_checkpoints()

print("\n========== RECURSIVE SEARCH RESULTS ==========")
print(f"Found {len(all_pth)} .pth files total")
for p in all_pth[:50]:
    print(" ", p)

print(f"\nFound {len(all_npy)} .npy files total")
for p in all_npy[:20]:
    print(" ", p)

print(f"\nCandidate checkpoint directories: {len(candidate_ckpt_dirs)}")
for d in candidate_ckpt_dirs[:20]:
    print(" ", d)

# -----------------------------
# HELPER: PICK FILES BY NAME
# -----------------------------
def pick_best_match(paths, preferred_names, fallback_keywords=None):
    """
    paths: list[Path]
    preferred_names: exact filenames in priority order
    fallback_keywords: list of keyword groups; each group is a tuple/list of strings
    """
    name_map = {p.name: p for p in paths}

    for fname in preferred_names:
        if fname in name_map:
            return name_map[fname]

    if fallback_keywords:
        for group in fallback_keywords:
            group = [g.lower() for g in group]
            matches = []
            for p in paths:
                n = p.name.lower()
                if all(g in n for g in group):
                    matches.append(p)
            if matches:
                matches = sorted(matches)
                return matches[0]

    return None

b0_path = pick_best_match(
    all_pth,
    preferred_names=["expert_1_efficientnet_b0.pth"],
    fallback_keywords=[("expert", "1", "b0"), ("efficientnet", "b0")]
)

b1_path = pick_best_match(
    all_pth,
    preferred_names=["expert_2_efficientnet_b1.pth"],
    fallback_keywords=[("expert", "2", "b1"), ("efficientnet", "b1")]
)

b2_path = pick_best_match(
    all_pth,
    preferred_names=["expert_3_efficientnet_b2.pth"],
    fallback_keywords=[("expert", "3", "b2"), ("efficientnet", "b2")]
)

student_path = pick_best_match(
    all_pth,
    preferred_names=["student_mobilenetv3_small.pth"],
    fallback_keywords=[("student", "mobilenet", "small"), ("student",)]
)

print("\n========== RESOLVED CHECKPOINTS ==========")
print("B0 path     :", b0_path)
print("B1 path     :", b1_path)
print("B2 path     :", b2_path)
print("Student path:", student_path)

missing = [name for name, p in {
    "expert_1_efficientnet_b0": b0_path,
    "expert_2_efficientnet_b1": b1_path,
    "expert_3_efficientnet_b2": b2_path,
    "student_mobilenetv3_small": student_path,
}.items() if p is None]

if missing:
    raise FileNotFoundError(
        "\nCould not find the required checkpoint files.\n\n"
        "Most likely reasons:\n"
        "1) You did NOT add the *saved output version* of the first Kaggle notebook as Input.\n"
        "2) In the first notebook, the artifact-saving cell was not run.\n"
        "3) You saved the notebook without including outputs.\n\n"
        "Expected files:\n"
        " - expert_1_efficientnet_b0.pth\n"
        " - expert_2_efficientnet_b1.pth\n"
        " - expert_3_efficientnet_b2.pth\n"
        " - student_mobilenetv3_small.pth\n\n"
        f"Missing: {missing}\n"
    )

# optional: infer checkpoint root
ckpt_dir = b0_path.parent
artifact_root = ckpt_dir.parent
print("\nArtifact root inferred as:", artifact_root)
print("Checkpoint dir inferred as:", ckpt_dir)

# -----------------------------
# MODEL BUILDERS
# -----------------------------
def build_expert(model_name: str, num_classes: int):
    if model_name == "efficientnet_b0":
        model = models.efficientnet_b0(weights=None)
    elif model_name == "efficientnet_b1":
        model = models.efficientnet_b1(weights=None)
    elif model_name == "efficientnet_b2":
        model = models.efficientnet_b2(weights=None)
    else:
        raise ValueError(model_name)

    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

def build_student(num_classes: int):
    model = models.mobilenet_v3_small(weights=None)
    in_features = model.classifier[3].in_features
    model.classifier[3] = nn.Linear(in_features, num_classes)
    return model

# -----------------------------
# LOAD MODELS
# -----------------------------
expert_b0 = build_expert("efficientnet_b0", NUM_CLASSES)
expert_b1 = build_expert("efficientnet_b1", NUM_CLASSES)
expert_b2 = build_expert("efficientnet_b2", NUM_CLASSES)
student = build_student(NUM_CLASSES)

expert_b0.load_state_dict(torch.load(b0_path, map_location=device))
expert_b1.load_state_dict(torch.load(b1_path, map_location=device))
expert_b2.load_state_dict(torch.load(b2_path, map_location=device))
student.load_state_dict(torch.load(student_path, map_location=device))

expert_b0.to(device).eval()
expert_b1.to(device).eval()
expert_b2.to(device).eval()
student.to(device).eval()

print("\nAll models loaded successfully.")

# -----------------------------
# PARAM COUNT
# -----------------------------
def count_params_m(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6

# -----------------------------
# LATENCY BENCHMARK
# -----------------------------
dummy = torch.randn(BATCH_SIZE, 3, IMAGE_SIZE, IMAGE_SIZE, device=device)

@torch.no_grad()
def measure_latency_ms(model, dummy_input, warmup=30, runs=200):
    model.eval()

    if device.type == "cuda":
        torch.cuda.synchronize()

    for _ in range(warmup):
        _ = model(dummy_input)

    if device.type == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()
    for _ in range(runs):
        _ = model(dummy_input)

    if device.type == "cuda":
        torch.cuda.synchronize()

    end = time.perf_counter()
    return (end - start) * 1000.0 / runs

lat_b0 = measure_latency_ms(expert_b0, dummy, WARMUP, RUNS)
lat_b1 = measure_latency_ms(expert_b1, dummy, WARMUP, RUNS)
lat_b2 = measure_latency_ms(expert_b2, dummy, WARMUP, RUNS)
lat_student = measure_latency_ms(student, dummy, WARMUP, RUNS)

forest_serial = lat_b0 + lat_b1 + lat_b2
forest_parallel_lb = max(lat_b0, lat_b1, lat_b2)

# -----------------------------
# RESULTS TABLE
# -----------------------------
latency_df = pd.DataFrame({
    "Model": [
        "Expert 1 (B0)",
        "Expert 2 (B1)",
        "Expert 3 (B2)",
        "Forest Teacher (serial)",
        "Forest Teacher (parallel lower bound)",
        "Student (Ours)",
    ],
    "Params (M)": [
        round(count_params_m(expert_b0), 2),
        round(count_params_m(expert_b1), 2),
        round(count_params_m(expert_b2), 2),
        round(count_params_m(expert_b0) + count_params_m(expert_b1) + count_params_m(expert_b2), 2),
        round(count_params_m(expert_b0) + count_params_m(expert_b1) + count_params_m(expert_b2), 2),
        round(count_params_m(student), 2),
    ],
    "Latency (ms)": [
        round(lat_b0, 2),
        round(lat_b1, 2),
        round(lat_b2, 2),
        round(forest_serial, 2),
        round(forest_parallel_lb, 2),
        round(lat_student, 2),
    ]
})

print("\n=== CONSISTENT LATENCY RESULTS ===")
print(latency_df.to_string(index=False))

# -----------------------------
# SAVE CSV
# -----------------------------
save_dir = Path("/kaggle/working/latency_rebenchmark")
save_dir.mkdir(parents=True, exist_ok=True)

out_csv = save_dir / "consistent_latency_results.csv"
latency_df.to_csv(out_csv, index=False)

print("\nSaved CSV to:", out_csv)

print("\nUse these in the paper:")
print(f"Expert 1 (B0): {lat_b0:.2f} ms")
print(f"Expert 2 (B1): {lat_b1:.2f} ms")
print(f"Expert 3 (B2): {lat_b2:.2f} ms")
print(f"Forest Teacher (serial): {forest_serial:.2f} ms")
print(f"Forest Teacher (parallel lower bound): {forest_parallel_lb:.2f} ms")
print(f"Student (Ours): {lat_student:.2f} ms")

Using device: cuda

========== DIRECTORY DEBUG ==========
Does /kaggle/input exist?    True
Does /kaggle/working exist?  True

Top-level folders under /kaggle/input:
  /kaggle/input/datasets
     - naifislam
  /kaggle/input/notebooks
     - abdullahrn

Top-level folders under /kaggle/working:
  /kaggle/working/.virtual_documents

========== RECURSIVE SEARCH RESULTS ==========
Found 4 .pth files total
  /kaggle/input/notebooks/abdullahrn/fork-of-effnet-1/saved_artifacts/checkpoints/expert_1_efficientnet_b0.pth
  /kaggle/input/notebooks/abdullahrn/fork-of-effnet-1/saved_artifacts/checkpoints/expert_2_efficientnet_b1.pth
  /kaggle/input/notebooks/abdullahrn/fork-of-effnet-1/saved_artifacts/checkpoints/expert_3_efficientnet_b2.pth
  /kaggle/input/notebooks/abdullahrn/fork-of-effnet-1/saved_artifacts/checkpoints/student_mobilenetv3_small.pth

Found 7 .npy files total
  /kaggle/input/notebooks/abdullahrn/fork-of-effnet-1/saved_artifacts/checkpoints/forest_weights.npy
  /kaggle/input/notebook

In [1]:
# ============================================================
# SINGLE CELL: Robustly find saved Kaggle notebook artifacts
# and regenerate confusion_combined_vertical.png
# with blue colormap like the paper figure
# ============================================================

import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# -----------------------------
# SETTINGS
# -----------------------------
CLASS_NAMES = [
    "Bicycle", "Bike", "Boat", "Bus", "Car", "Cng", "Easy-bike",
    "Horse-cart", "Leguna", "Rickshaw", "Tractor", "Truck",
    "Van", "Wheelbarrow"
]

FIGSIZE = (12, 18)
DPI = 300
CMAP = "Blues"

input_root = Path("/kaggle/input")
working_root = Path("/kaggle/working")

print("========== DIRECTORY DEBUG ==========")
print("Does /kaggle/input exist?   ", input_root.exists())
print("Does /kaggle/working exist? ", working_root.exists())

# -----------------------------
# RECURSIVE SEARCH
# -----------------------------
def recursive_find_prediction_files():
    search_roots = []
    if input_root.exists():
        search_roots.append(input_root)
    if working_root.exists():
        search_roots.append(working_root)

    all_npy = []
    for root in search_roots:
        try:
            all_npy.extend(root.rglob("*.npy"))
        except Exception as e:
            print(f"Could not scan {root}: {e}")

    return sorted(all_npy)

all_npy = recursive_find_prediction_files()

print("\n========== RECURSIVE SEARCH RESULTS ==========")
print(f"Found {len(all_npy)} .npy files total")
for p in all_npy[:50]:
    print(" ", p)

# -----------------------------
# HELPER: PICK FILES BY NAME
# -----------------------------
def pick_best_match(paths, preferred_names, fallback_keywords=None):
    name_map = {}
    for p in paths:
        name_map.setdefault(p.name, p)

    for fname in preferred_names:
        if fname in name_map:
            return name_map[fname]

    if fallback_keywords:
        for group in fallback_keywords:
            group = [g.lower() for g in group]
            matches = []
            for p in paths:
                n = p.name.lower()
                if all(g in n for g in group):
                    matches.append(p)
            if matches:
                matches = sorted(matches)
                return matches[0]

    return None

y_true_path = pick_best_match(
    all_npy,
    preferred_names=["y_true.npy"],
    fallback_keywords=[("y", "true")]
)

forest_pred_path = pick_best_match(
    all_npy,
    preferred_names=["forest_teacher.npy"],
    fallback_keywords=[("forest", "teacher")]
)

student_pred_path = pick_best_match(
    all_npy,
    preferred_names=["student_ours.npy"],
    fallback_keywords=[("student", "ours"), ("student",)]
)

print("\n========== RESOLVED FILES ==========")
print("y_true        :", y_true_path)
print("forest_teacher:", forest_pred_path)
print("student_ours  :", student_pred_path)

missing = [name for name, p in {
    "y_true.npy": y_true_path,
    "forest_teacher.npy": forest_pred_path,
    "student_ours.npy": student_pred_path,
}.items() if p is None]

if missing:
    raise FileNotFoundError(
        "\nCould not find the required prediction files.\n\n"
        "Expected files:\n"
        " - y_true.npy\n"
        " - forest_teacher.npy\n"
        " - student_ours.npy\n\n"
        f"Missing: {missing}\n"
    )

# -----------------------------
# LOAD ARRAYS
# -----------------------------
y_true = np.load(y_true_path)
forest_pred = np.load(forest_pred_path)
student_pred = np.load(student_pred_path)

print("\nLoaded arrays successfully.")
print("y_true shape        :", y_true.shape)
print("forest_pred shape   :", forest_pred.shape)
print("student_pred shape  :", student_pred.shape)

# -----------------------------
# CONFUSION MATRICES
# -----------------------------
labels = list(range(len(CLASS_NAMES)))

cm_forest = confusion_matrix(y_true, forest_pred, labels=labels, normalize="true")
cm_student = confusion_matrix(y_true, student_pred, labels=labels, normalize="true")

# -----------------------------
# PLOT FUNCTION
# -----------------------------
def draw_confmat(ax, cm, class_names, title, cmap="Blues"):
    im = ax.imshow(cm, interpolation="nearest", cmap=cmap, vmin=0.0, vmax=1.0)
    ax.set_title(title, fontsize=16, pad=10)
    ax.set_xticks(np.arange(len(class_names)))
    ax.set_yticks(np.arange(len(class_names)))
    ax.set_xticklabels(class_names, rotation=45, ha="right", fontsize=10)
    ax.set_yticklabels(class_names, fontsize=10)
    ax.set_xlabel("Predicted", fontsize=12)
    ax.set_ylabel("True", fontsize=12)

    thresh = 0.50
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            val = cm[i, j]
            ax.text(
                j, i, f"{val:.2f}",
                ha="center", va="center",
                color="white" if val > thresh else "black",
                fontsize=8
            )
    return im

# -----------------------------
# MAKE COMBINED VERTICAL FIGURE
# -----------------------------
fig, axes = plt.subplots(2, 1, figsize=FIGSIZE)

fig.suptitle("Forest Teacher (Normalized)", fontsize=22, y=0.98)

im1 = draw_confmat(axes[0], cm_forest, CLASS_NAMES, "Forest Teacher (Normalized)", cmap=CMAP)
im2 = draw_confmat(axes[1], cm_student, CLASS_NAMES, "Student (Ours) (Normalized)", cmap=CMAP)

# Separate colorbars for each subplot, like your shown figure
cbar1 = fig.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)
cbar1.ax.tick_params(labelsize=10)

cbar2 = fig.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)
cbar2.ax.tick_params(labelsize=10)

plt.tight_layout(rect=[0, 0, 1, 0.97])

# -----------------------------
# SAVE OUTPUTS
# -----------------------------
save_dir = Path("/kaggle/working/confusion_matrix_regen")
save_dir.mkdir(parents=True, exist_ok=True)

out_png = save_dir / "confusion_combined_vertical.png"
out_pdf = save_dir / "confusion_combined_vertical.pdf"

fig.savefig(out_png, dpi=DPI, bbox_inches="tight")
fig.savefig(out_pdf, bbox_inches="tight")
plt.close(fig)

print("\nSaved:")
print(" ", out_png)
print(" ", out_pdf)

print("\nDone. Use this new file in Overleaf:")
print("confusion_combined_vertical.png")

========== DIRECTORY DEBUG ==========
Does /kaggle/input exist?    True
Does /kaggle/working exist?  True

========== RECURSIVE SEARCH RESULTS ==========
Found 7 .npy files total
  /kaggle/input/notebooks/abdullahrn/fork-of-effnet-1/saved_artifacts/checkpoints/forest_weights.npy
  /kaggle/input/notebooks/abdullahrn/fork-of-effnet-1/saved_artifacts/predictions/expert_1_b0.npy
  /kaggle/input/notebooks/abdullahrn/fork-of-effnet-1/saved_artifacts/predictions/expert_2_b1.npy
  /kaggle/input/notebooks/abdullahrn/fork-of-effnet-1/saved_artifacts/predictions/expert_3_b2.npy
  /kaggle/input/notebooks/abdullahrn/fork-of-effnet-1/saved_artifacts/predictions/forest_teacher.npy
  /kaggle/input/notebooks/abdullahrn/fork-of-effnet-1/saved_artifacts/predictions/student_ours.npy
  /kaggle/input/notebooks/abdullahrn/fork-of-effnet-1/saved_artifacts/predictions/y_true.npy

========== RESOLVED FILES ==========
y_true        : /kaggle/input/notebooks/abdullahrn/fork-of-effnet-1/saved_artifacts/predictions

In [2]:
# ============================================================
# SINGLE CELL: Generate the 4 paper plots only
# Output names are kept EXACTLY the same:
#   - plot_val_accuracy.png
#   - plot_test_accuracy.png
#   - plot_params.png
#   - panel_d_gmacs_latency.png
# ============================================================

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ------------------------------------------------------------
# SAVE LOCATION
# ------------------------------------------------------------
OUT_DIR = Path("/kaggle/working/saved_artifacts/plots")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# USE THE VALUES SHOWN IN YOUR CURRENT PLOTS
# ------------------------------------------------------------
T = np.arange(0, 601)

models = [
    "Expert_1 (B0)",
    "Expert_2 (B1)",
    "Expert_3 (B2)",
    "Forest_Teacher",
    "Student (MobileNetV3)",
]

colors = {
    "Expert_1 (B0)": "tab:blue",
    "Expert_2 (B1)": "tab:orange",
    "Expert_3 (B2)": "tab:green",
    "Forest_Teacher": "tab:red",
    "Student (MobileNetV3)": "tab:purple",
}

linestyles = {
    "Expert_1 (B0)": "-",
    "Expert_2 (B1)": "--",
    "Expert_3 (B2)": "-.",
    "Forest_Teacher": ":",
    "Student (MobileNetV3)": (0, (3, 1, 1, 1)),
}

# (a) Validation Accuracy
val_acc = {
    "Expert_1 (B0)": 94.74,
    "Expert_2 (B1)": 92.51,
    "Expert_3 (B2)": 95.09,
    "Forest_Teacher": 96.52,
    "Student (MobileNetV3)": 91.53,
}

# (b) Test Accuracy
test_acc = {
    "Expert_1 (B0)": 95.10,
    "Expert_2 (B1)": 93.58,
    "Expert_3 (B2)": 95.01,
    "Forest_Teacher": 95.99,
    "Student (MobileNetV3)": 91.00,
}

# (c) Params (M)
params_m = {
    "Expert_1 (B0)": 4.03,
    "Expert_2 (B1)": 6.53,
    "Expert_3 (B2)": 7.72,
    "Forest_Teacher": 18.28,
    "Student (MobileNetV3)": 1.53,
}

# (d) GMACs + Latency (ms)
gmacs = {
    "Expert_1 (B0)": 0.409,
    "Expert_2 (B1)": 0.603,
    "Expert_3 (B2)": 0.694,
    "Forest_Teacher": 1.705,
    "Student (MobileNetV3)": 0.060,
}
latency_ms = {
    "Expert_1 (B0)": 8.14,
    "Expert_2 (B1)": 11.63,
    "Expert_3 (B2)": 12.54,
    "Forest_Teacher": 32.31,
    "Student (MobileNetV3)": 5.34,
}

# ------------------------------------------------------------
# STYLE
# ------------------------------------------------------------
plt.style.use("ggplot")
plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 18,
    "axes.labelsize": 14,
    "legend.fontsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
})

def plot_scalar_lines(values_dict, ylabel, title, out_path, legend_fmt):
    fig, ax = plt.subplots(figsize=(12, 6), dpi=200)

    legend_handles = []
    legend_labels = []

    for m in models:
        y = np.full_like(T, values_dict[m], dtype=float)
        ax.plot(
            T, y,
            color=colors[m],
            linestyle=linestyles[m],
            linewidth=2.5
        )
        ax.text(
            T[-1] + 15, values_dict[m], f"{values_dict[m]:.2f}",
            va="center", ha="left", fontsize=12, color="black"
        )
        legend_handles.append(
            Line2D([0], [0], color=colors[m], linestyle=linestyles[m], linewidth=2.5)
        )
        legend_labels.append(legend_fmt(m, values_dict[m]))

    ax.set_xlim(0, 650)
    ax.set_xticks(np.arange(0, 601, 100))
    ax.set_xlabel("stream t")
    ax.set_ylabel(ylabel)
    ax.set_title(title, pad=8)
    ax.grid(True, alpha=0.5)

    ax.legend(
        legend_handles, legend_labels,
        loc="upper center", bbox_to_anchor=(0.5, 1.25),
        ncol=2, frameon=True
    )

    plt.tight_layout()
    plt.savefig(out_path, bbox_inches="tight")
    plt.close()
    print(f"Saved: {out_path}")

# ------------------------------------------------------------
# (a) Validation Accuracy
# ------------------------------------------------------------
plot_scalar_lines(
    values_dict=val_acc,
    ylabel="Accuracy (%)",
    title="(a) Validation Accuracy",
    out_path=OUT_DIR / "plot_val_accuracy.png",
    legend_fmt=lambda m, v: f"{m} ({v:.2f}%)"
)

# ------------------------------------------------------------
# (b) Test Accuracy
# ------------------------------------------------------------
plot_scalar_lines(
    values_dict=test_acc,
    ylabel="Accuracy (%)",
    title="(b) Test Accuracy",
    out_path=OUT_DIR / "plot_test_accuracy.png",
    legend_fmt=lambda m, v: f"{m} ({v:.2f}%)"
)

# ------------------------------------------------------------
# (c) Model Size (Params)
# ------------------------------------------------------------
plot_scalar_lines(
    values_dict=params_m,
    ylabel="Params (M)",
    title="(c) Model Size (Params)",
    out_path=OUT_DIR / "plot_params.png",
    legend_fmt=lambda m, v: f"{m} ({v:.2f} M)"
)

# ------------------------------------------------------------
# (d) Compute (GMACs) and Latency (ms)
# ------------------------------------------------------------
fig, ax1 = plt.subplots(figsize=(12, 7), dpi=200)
ax2 = ax1.twinx()

legend_handles = []
legend_labels = []

for m in models:
    # GMAC lines (solid)
    ax1.plot(
        T, np.full_like(T, gmacs[m], dtype=float),
        color=colors[m], linestyle="-", linewidth=2.5
    )
    legend_handles.append(
        Line2D([0], [0], color=colors[m], linestyle="-", linewidth=2.5)
    )
    legend_labels.append(f"{m} (GMACs={gmacs[m]:.3f})")

for m in models:
    # Latency lines (dashed)
    ax2.plot(
        T, np.full_like(T, latency_ms[m], dtype=float),
        color=colors[m], linestyle="--", linewidth=2.5
    )
    legend_handles.append(
        Line2D([0], [0], color=colors[m], linestyle="--", linewidth=2.5)
    )
    legend_labels.append(f"{m} (Latency={latency_ms[m]:.2f} ms)")

ax1.set_xlim(0, 600)
ax1.set_xticks(np.arange(0, 601, 100))
ax1.set_xlabel("stream t")
ax1.set_ylabel("GMACs")
ax2.set_ylabel("Latency (ms)")
ax1.set_title("(d) Compute (GMACs) and Latency (ms)", pad=8)

ax1.grid(True, alpha=0.5)

fig.legend(
    legend_handles, legend_labels,
    loc="upper center", bbox_to_anchor=(0.5, 1.12),
    ncol=2, frameon=True, fontsize=11
)

plt.tight_layout()
plt.savefig(OUT_DIR / "panel_d_gmacs_latency.png", bbox_inches="tight")
plt.close()
print(f"Saved: {OUT_DIR / 'panel_d_gmacs_latency.png'}")

print("\nDone.")
print("Files generated in:", OUT_DIR)

Saved: /kaggle/working/saved_artifacts/plots/plot_val_accuracy.png
Saved: /kaggle/working/saved_artifacts/plots/plot_test_accuracy.png
Saved: /kaggle/working/saved_artifacts/plots/plot_params.png
Saved: /kaggle/working/saved_artifacts/plots/panel_d_gmacs_latency.png

Done.
Files generated in: /kaggle/working/saved_artifacts/plots
